In [18]:
import os
import pandas as pd

df = pd.read_csv("nyc_taxi/yellow_tripdata_2015-01.csv")

# 1. EDA

In [19]:
# 1. size
print("size(row, col):", df.shape)
# 2. type
print("\ncol name and type:")
print(df.dtypes)

size(row, col): (12748986, 19)

col name and type:
VendorID                   int64
tpep_pickup_datetime         str
tpep_dropoff_datetime        str
passenger_count            int64
trip_distance            float64
pickup_longitude         float64
pickup_latitude          float64
RateCodeID                 int64
store_and_fwd_flag           str
dropoff_longitude        float64
dropoff_latitude         float64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
dtype: object


In [21]:
print("Earliest pick-up time: ", df["tpep_pickup_datetime"].min())
print("Latest pick-up time: ", df["tpep_pickup_datetime"].max())

Earliest pick-up time:  2015-01-01 00:00:00
Latest pick-up time:  2015-01-31 23:59:59


Data Cleaning

In [25]:
n_before = len(df)
print("Row number before clean:", n_before)

Row number before clean: 12748986


In [26]:
lat_min, lat_max = 40.5, 41.0      
lon_min, lon_max = -74.3, -73.7   

mask = (
    (df["pickup_latitude"].between(lat_min, lat_max)) &
    (df["pickup_longitude"].between(lon_min, lon_max))
)
df_clean = df[mask].copy()

n_after = len(df_clean)
print("Row number after clean:", n_after)
print("Number cleaned:", n_before - n_after)
print("Proprotion cleaned: {:.2%}".format((n_before - n_after) / n_before))

Row number after clean: 12503700
Number cleaned: 245286
Proprotion cleaned: 1.92%


# 2. H3 Transformation

In [28]:
import sys
!{sys.executable} -m pip install h3

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.1/848.1 kB 7.8 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [30]:
import h3
print("h3 version:", h3.__version__)

# Try on a single point
lat, lng = 40.758, -73.985
resolution = 9  

cell = h3.latlng_to_cell(lat, lng, resolution)
print("H3 number:", cell)

h3 version: 4.5.0
H3 number: 892a100d67bffff


In [ ]:
# H3 conversion
resolution = 9

df_clean["h3_str"] = [
    h3.latlng_to_cell(lat, lng, resolution)
    for lat, lng in zip(df_clean["pickup_latitude"], df_clean["pickup_longitude"])
]

df_clean["h3_id"] = df_clean["h3_str"].apply(h3.str_to_int)

print(df_clean[["pickup_longitude", "pickup_latitude", "h3_str", "h3_id"]].head())

print("\nh3_id type:", df_clean["h3_id"].dtype)

   pickup_longitude  pickup_latitude           h3_str               h3_id
0        -73.993896        40.750111  892a100d2cbffff  617733123811835903
1        -74.001648        40.724243  892a1072c03ffff  617733151078481919
2        -73.963341        40.802788  892a1008897ffff  617733122566914047
3        -74.009087        40.713818  892a1072c67ffff  617733151085035519
4        -73.971176        40.762428  892a100d613ffff  617733123866886143

h3_id type: int64


In [ ]:
# Check the transformation
sample = df_clean.head(5).copy()
sample["h3_center"] = sample["h3_str"].apply(lambda c: h3.cell_to_latlng(c))
print(sample[["pickup_latitude", "pickup_longitude", "h3_center"]])

   pickup_latitude  pickup_longitude                                 h3_center
0        40.750111        -73.993896   (40.74969237930922, -73.99427966126574)
1        40.724243        -74.001648    (40.7230543642727, -74.00178736299634)
2        40.802788        -73.963341   (40.80345319173592, -73.96211573734033)
3        40.713818        -74.009087   (40.714959860432664, -74.0078835084681)
4        40.762428        -73.971176  (40.763533802293004, -73.97127552650305)


In [ ]:
# Print the final table
final_cols = [
    "tpep_pickup_datetime",   
    "pickup_longitude",       # lon(float64 → double)
    "pickup_latitude",        # lat(float64 → double)
    "h3_id",                  # H3(int64 → bigint)
]
df_final = df_clean[final_cols].copy()

print("shape:", df_final.shape)
print("\ntype:")
print(df_final.dtypes)
print("\nfirst 5 row:")
print(df_final.head())

shape: (12503700, 4)

type:
tpep_pickup_datetime        str
pickup_longitude        float64
pickup_latitude         float64
h3_id                     int64
dtype: object

first 5 row:
  tpep_pickup_datetime  pickup_longitude  pickup_latitude               h3_id
0  2015-01-15 19:05:39        -73.993896        40.750111  617733123811835903
1  2015-01-10 20:33:38        -74.001648        40.724243  617733151078481919
2  2015-01-10 20:33:38        -73.963341        40.802788  617733122566914047
3  2015-01-10 20:33:39        -74.009087        40.713818  617733151085035519
4  2015-01-10 20:33:39        -73.971176        40.762428  617733123866886143
